# Toy Models of Superposition - a hands-on intro to mechanistic interpretability

We're going to partially replicate the paper [Toy Models of Superposition](https://transformer-circuits.pub/2022/toy_model/index.html) (Elhage et al., Anthropic, 2022). It's the friendliest paper in mechanistic interpretability because the "model" is tiny - a single linear layer down to a bottleneck and back - yet it cleanly demonstrates a phenomenon that drives a lot of modern interp research.

## What is mechanistic interpretability?

Mech interp is the project of reverse-engineering neural networks - opening them up and figuring out what algorithm they actually learnt, in terms a human can understand. Think "decompiler for neural nets." Neel Nanda is one of the most prolific researchers and teachers in this field.

## What is superposition, in one sentence?

Superposition is when a neural network represents more features than it has neurons, by packing them into non-orthogonal directions in activation space.

If a model has 100 hidden neurons, it doesn't represent 100 features - it represents thousands of features, each as a slightly-overlapping direction in the 100-dimensional hidden space. The directions can't be orthogonal (you can only fit 100 orthogonal directions in 100D), so features interfere with each other. The network tolerates this interference because, in practice, features are sparse - most features are zero in any given input, so collisions are rare.

This matters for interpretability because it means individual neurons are usually polysemantic - one neuron responds to many unrelated features at once. To understand what a model is doing, you can't just look at neurons; you have to find the directions that correspond to single concepts. This is exactly what sparse autoencoders (SAEs, the hot topic in 2024+ interp) are trying to do.

We're going to see superposition emerge in a tiny model we train from scratch.

## 1. Setup

On Colab: `Runtime → Change runtime type → GPU` (a T4 is fine; this is a tiny model and would work on CPU too).

PyTorch and matplotlib are pre-installed on Colab, so the imports should just work.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

torch.manual_seed(0)
np.random.seed(0)

## 2. The model

Here's the model. It is absurdly small - just a single weight matrix `W` of shape `(m, n)` where `n` is the number of input features and `m < n` is the bottleneck dimension.

The forward pass is:
1. Encode: `h = W @ x` - project the `n`-dim input down to `m` dims (this is the lossy bottleneck).
2. Decode: `out = W.T @ h + b` - project back up to `n` dims using the same weights transposed (a 'tied autoencoder').
3. ReLU at the end - the model is only allowed to output non-negative values.

The ReLU is the secret sauce - it lets the model represent more than `m` features in the `m`-dim hidden space, because it can throw away the negative-interference from other features. Without the ReLU, this would just be PCA.

PyTorch primer (in case you're new): `nn.Module` is the base class for all neural networks. `nn.Parameter` tells PyTorch "this tensor is a learnable parameter - track gradients for it and let the optimizer update it."

In [ ]:
class ToyModel(nn.Module):
    def __init__(self, n_features: int, n_hidden: int):
        super().__init__()
        # W has shape (n_hidden, n_features). Each column is the 'direction' the model uses
        # to represent that feature in hidden space.
        self.W = nn.Parameter(torch.empty(n_hidden, n_features))
        nn.init.xavier_normal_(self.W)
        self.b = nn.Parameter(torch.zeros(n_features))

    def forward(self, x):
        # x: (batch, n_features)
        hidden = x @ self.W.T          # (batch, n_hidden) -- encode
        out = hidden @ self.W + self.b # (batch, n_features) -- decode (tied weights)
        return F.relu(out)

## 3. Synthetic data - sparse features

Real-world features (like "this token is in a Python comment" or "this sentence is about a dog") are sparse - only a tiny fraction of features are active for any given input. We simulate this directly.

For each input vector:
- Each of the `n` features is independently active with probability `(1 - S)`, where `S` is the sparsity.
- When active, the feature value is drawn from `Uniform(0, 1)`.
- When inactive, the value is exactly `0`.

So `S = 0` → every feature is always on (dense). `S = 0.99` → only ~1% of features are active per input (very sparse, much like real data).

In [ ]:
def generate_batch(batch_size: int, n_features: int, sparsity: float, device):
    """Returns a (batch_size, n_features) tensor of sparse non-negative features."""
    values = torch.rand(batch_size, n_features, device=device)
    mask   = torch.rand(batch_size, n_features, device=device) > sparsity
    return values * mask

# sanity check: at S=0.9, roughly 10% of entries should be nonzero
sample = generate_batch(1000, n_features=5, sparsity=0.9, device=device)
print(f'Fraction of nonzero entries: {(sample > 0).float().mean().item():.3f} (expected ~0.10)')

## 4. Feature importance and loss

Not all features matter equally. Following the paper, we give each feature an importance `I_i` that decays geometrically: feature 0 is the most important, feature `n-1` the least.

The loss is importance-weighted MSE between input and reconstruction. Important features pay a bigger penalty when reconstructed poorly, so the model will preferentially represent them.

This is the lever that creates interesting behaviour at low sparsity: with `n=5, m=2` and no sparsity, the model can only represent the 2 most important features (it's basically doing PCA). But with sparsity, it can squeeze all 5 in.

In [ ]:
def make_importance(n_features: int, decay: float = 0.7, device='cpu'):
    return decay ** torch.arange(n_features, device=device, dtype=torch.float32)

def loss_fn(out, target, importance):
    # out, target: (batch, n_features); importance: (n_features,)
    return ((out - target) ** 2 * importance).mean()

print('Importance weights (n=5):', make_importance(5).numpy())

## 5. Training loop

Standard PyTorch training. We use AdamW (a popular optimizer - for now, just trust that it works well as a default). Each step:
1. Generate a fresh batch of synthetic sparse data.
2. Run it through the model.
3. Compute the importance-weighted reconstruction loss.
4. Backprop and update the weights.

We use a cosine learning rate schedule that starts at `1e-3` and decays to near zero. This is just a quality-of-life trick - it makes the final loss noticeably lower and the visualisations cleaner. Don't worry about why it helps; for now treat it as a recipe.

In [ ]:
def train(n_features=5, n_hidden=2, sparsity=0.0, n_steps=10_000, batch_size=1024,
          lr=1e-3, verbose=False):
    model = ToyModel(n_features=n_features, n_hidden=n_hidden).to(device)
    importance = make_importance(n_features, device=device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_steps)

    losses = []
    for step in range(n_steps):
        x = generate_batch(batch_size, n_features, sparsity, device=device)
        out = model(x)
        loss = loss_fn(out, x, importance)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()

        losses.append(loss.item())
        if verbose and step % 1000 == 0:
            print(f'step {step:5d}  loss {loss.item():.5f}')

    return model, losses

### Smoke test: train one model and check the loss curve

Let's verify training works at all. We expect to see the loss drop by ~2 orders of magnitude over training.

In [ ]:
model, losses = train(sparsity=0.9, verbose=True)

plt.figure(figsize=(6, 3))
plt.plot(losses)
plt.yscale('log')
plt.xlabel('step')
plt.ylabel('loss (log scale)')
plt.title('Training loss (sparsity=0.9)')
plt.grid(True, alpha=0.3)
plt.show()

## 6. The headline experiment: sweep over sparsity

Now the interesting bit. We'll train a fresh model at each of several sparsity levels, then plot the columns of `W` as arrows in 2D space (since `n_hidden=2`, we can draw `W` directly on the page).

What each arrow means: column `i` of `W` is the 2D direction the model uses to represent feature `i`. The length of the arrow tells you how strongly the model represents that feature; the angle between two arrows tells you how much those features interfere with each other.

What to look for:
- At `S=0` (no sparsity): only the 2 most important features get represented (as orthogonal axes). The other 3 columns collapse to ~zero length. This is essentially PCA.
- As `S` increases: the model starts squeezing in more features by placing them at non-orthogonal angles.
- At `S=0.99`: all 5 features arrange as a regular pentagon - 5 equally-spaced directions in the 2D plane. This is superposition!


In [ ]:
def plot_W(W, ax, title=''):
    """W is shape (2, n_features). Draw each column as an arrow from origin."""
    colors = plt.cm.viridis(np.linspace(0, 1, W.shape[1]))
    for i in range(W.shape[1]):
        ax.arrow(0, 0, W[0, i], W[1, i],
                 head_width=0.04, length_includes_head=True,
                 color=colors[i], alpha=0.9, linewidth=2)
    lim = 1.3
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
    ax.set_aspect('equal')
    ax.axhline(0, color='gray', linewidth=0.5)
    ax.axvline(0, color='gray', linewidth=0.5)
    ax.set_title(title)
    ax.grid(True, alpha=0.3)

In [ ]:
sparsities = [0.0, 0.7, 0.9, 0.97, 0.99]
fig, axes = plt.subplots(1, len(sparsities), figsize=(4 * len(sparsities), 4))

trained_models = {}
for ax, S in zip(axes, sparsities):
    model, _ = train(n_features=5, n_hidden=2, sparsity=S, n_steps=10_000)
    trained_models[S] = model
    W = model.W.detach().cpu().numpy()
    plot_W(W, ax, title=f'sparsity S = {S}')

plt.suptitle('Columns of W (= feature directions) at increasing sparsity', y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

## 7. Discussion: what just happened?

Left plot (`S=0`): only 2 long arrows, perpendicular. The model used its 2 hidden dimensions to perfectly store the 2 most important features and ignored the other 3. This is the regime where superposition isn't worth it - features are always active, so any non-orthogonality would cause constant interference.

Middle plots (`S=0.7`, `0.9`): some intermediate configuration - often a triangle with one or two extra arrows starting to push in.

Right plot (`S=0.99`): a regular pentagon! All 5 features are represented as 5 equally-spaced directions. Pairs of features have a 72° angle between them, so there's some interference - when feature 1 is active, the model will leak a small positive value into the feature 2 output direction. But since features are 99% sparse, two features almost never co-occur, and the model's ReLU can clip away small positive leakage. Net result: the model successfully represents 5 features in 2 dimensions.

This is superposition. And it's the headline phenomenon driving a huge chunk of modern interpretability research:
- It explains why individual neurons in real models are polysemantic.
- It motivates sparse autoencoders (SAEs), which try to recover the original features from a superposed hidden state by training a wider autoencoder with a sparsity penalty.
- It connects to compressed sensing, the Johnson–Lindenstrauss lemma, and other ideas from classical maths.

Compare your plot side-by-side with [Figure 7 of the original paper](https://transformer-circuits.pub/2022/toy_model/index.html#geometry). You should see essentially the same pentagon.

## 8. Stretch goal: feature dimensionality

The paper introduces a per-feature quantity called dimensionality:

$$Di = \frac{\lVert Wi \rVert^2}{\sumj (\hat{Wi} \cdot W_j)^2}$$

where $Wi$ is column $i$ of $W$ and $\hat{Wi}$ is its unit vector. Intuitively, $D_i$ measures "how much of a hidden dimension does this feature get to itself?" It's between 0 and 1:

- $D_i = 1$: feature has its own orthogonal axis (no interference).
- $D_i = 1/2$: feature shares a dimension with one other (antipodal pair).
- $D_i = 2/3$: triangle.
- $D_i = 2/5$: pentagon vertex.
- $D_i = 0$: feature isn't represented at all.

If we average dimensionality across all features and sweep over many sparsity levels, we get a striking plot with discrete plateaus at these rational values - sharp phase transitions between geometries. This is one of the most beautiful figures in the paper.

In [ ]:
def feature_dimensionalities(model):
    """Return per-feature dimensionality D_i."""
    W = model.W.detach()                           # (m, n)
    norms = W.norm(dim=0)                          # (n,)
    eps = 1e-8
    unit = W / (norms + eps)                       # (m, n)
    overlap = (unit.T @ W) ** 2                    # (n, n): (W_hat_i . W_j)^2
    denom = overlap.sum(dim=1)                     # sum over j
    return (norms ** 2) / (denom + eps)

# sweep a denser range of sparsities; small model so each train is quick
log_one_minus_S = np.linspace(np.log10(1.0), np.log10(0.005), 25)
sparsity_grid = 1 - 10 ** log_one_minus_S

mean_dims = []
for S in sparsity_grid:
    model, _ = train(n_features=5, n_hidden=2, sparsity=float(S), n_steps=5_000)
    D = feature_dimensionalities(model).cpu().numpy()
    mean_dims.append(D)

mean_dims = np.array(mean_dims)  # (n_sparsities, n_features)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

# plot every feature, plus the mean
for i in range(mean_dims.shape[1]):
    ax.plot(1 - sparsity_grid, mean_dims[:, i], 'o-', alpha=0.3, label=f'feat {i}' if i < 5 else None)
ax.plot(1 - sparsity_grid, mean_dims.mean(axis=1), 'k-', linewidth=2.5, label='mean')

# horizontal lines at the geometric plateaus
for y, name in [(1.0, 'orthogonal'), (2/3, 'triangle (2/3)'),
                (1/2, 'antipodal (1/2)'), (2/5, 'pentagon (2/5)')]:
    ax.axhline(y, color='red', linestyle='--', linewidth=0.7, alpha=0.6)
    ax.text(0.0015, y + 0.01, name, color='red', fontsize=8)

ax.set_xscale('log')
ax.set_xlabel('feature density (1 - sparsity)')
ax.set_ylabel('feature dimensionality D_i')
ax.set_title('Phase transitions in feature geometry as sparsity changes')
ax.invert_xaxis()
ax.legend(loc='lower right', fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Where to go from here

You've just reproduced the core finding of one of the most influential mech interp papers of the last few years. Some directions if you want to keep going:

Stay in the toy model:
- Vary `n_features` and `n_hidden` (e.g. `n=20, m=5`) and see what geometries appear.
- Implement the "absolute value" variant from the paper (Section 4) - features are signed, model outputs `|.|`. Different geometries emerge.
- Add a small L1 penalty on the hidden activations and see how it changes the learnt `W`.

Step up to real models:
- Read Neel Nanda's [TransformerLens tutorial](https://neelnanda-io.github.io/TransformerLens/) and load a pre-trained GPT-2 small.
- Replicate Anthropic's [induction heads](https://transformer-circuits.pub/2021/framework/index.html) finding in a 2-layer attention-only transformer.
- Try training a sparse autoencoder on the residual stream of GPT-2 small - this is the natural follow-up to this notebook and the current frontier of interp research.

Reading list:
- Neel Nanda's [Getting Started in Mech Interp](https://www.neelnanda.io/mechanistic-interpretability/getting-started) - the canonical entry point.
- Anthropic's [Toy Models of Superposition](https://transformer-circuits.pub/2022/toy_model/index.html) - what we just did, plus much more.
- Anthropic's [Scaling Monosemanticity](https://transformer-circuits.pub/2024/scaling-monosemanticity/) - what SAEs look like on a real production-scale model.
